# Matriz de consistencia

Tesis: Modernización de una arquitectura de dos flujos para la detección de anomalías en videos de vigilancia mediante backbones convolucionales recientes, representaciones de movimiento y mecanismos de atención temporal.

Enfoque. El trabajo es experimental: el objetivo es medirse contra métodos publicados que usan UCF-Crime. Como esos artículos trabajan con 13 clases y otra resolución, sus cifras no se pueden comparar directo. Por eso Sultani, RTFM y UR-DMU se reentrenan con nuestro protocolo: los mismos videos a 320×240, las mismas tres clases (Normal, Aggression y Theft), las mismas particiones (train 390, val 98, test 203) y cinco semillas. Cada método usa las features de su artículo (I3D) y se le agrega solo una cabeza de tres clases. El modelo se elige con validación y el test se usa una sola vez.

## 1. Matriz

| Problema | Objetivo | Hipótesis | Variables | Indicadores |
|---|---|---|---|---|
| General: ¿En qué medida una arquitectura de dos flujos con backbones convolucionales (ConvNeXt-S en apariencia y ConvNeXt-T sobre la diferencia de frames) y un Transformer temporal entrenado con MIL mejora la clasificación de videos de vigilancia de UCF-Crime en Normal, Aggression y Theft, y la ubicación del evento en el tiempo, frente a métodos de referencia reentrenados con el mismo protocolo? | Diseñar, implementar y evaluar la arquitectura propuesta y compararla con Sultani et al., RTFM y UR-DMU, reentrenados con el mismo split, la misma resolución (320×240) y las mismas tres clases. | El modelo propuesto obtiene un F1-macro mayor que cada método de referencia y un AUC por frame igual o mayor. | VI: arquitectura (propuesta, Sultani, RTFM, UR-DMU, regresión logística). VD: desempeño de clasificación por video y de localización temporal. | F1-macro con IC 95 %, McNemar, AUC y AP por frame. |
| PE1: ¿El modelo propuesto clasifica los videos mejor que los métodos MIL de referencia? | OE1: Medir la clasificación por video del modelo propuesto y compararla con Sultani, RTFM y UR-DMU. | HE1: El F1-macro del modelo propuesto es mayor que el de cada método de referencia. H0: ΔF1-macro ≤ 0. H1: ΔF1-macro > 0. | VI: arquitectura. VD: desempeño de clasificación por video. | F1-macro, F1 por clase, F1-anomalía, AUC y AP por video, matriz de confusión; bootstrap pareado y McNemar. |
| PE2: ¿Qué tan bien ubica el modelo propuesto el evento dentro del video? | OE2: Medir la localización temporal con las anotaciones oficiales del test y compararla con los métodos de referencia. | HE2: El AUC por frame del modelo propuesto es igual o mayor que el de los métodos de referencia. H0: ΔAUC frame < 0. H1: ΔAUC frame ≥ 0. | VI: arquitectura. VD: desempeño de localización temporal. | AUC y AP por frame, tIoU medio, porcentaje de eventos con tIoU ≥ 0,1 / 0,3 / 0,5, falsas alarmas en videos normales. |
| PE3: ¿Cuánto aporta el modelado temporal (Transformer con MIL) frente a un clasificador sin modelado temporal sobre las mismas características? | OE3: Comparar el modelo propuesto con una regresión logística entrenada sobre las mismas características de los dos flujos. | HE3: El Transformer con MIL supera a la regresión logística en F1-macro y en AUC por frame. H0: Δ ≤ 0. H1: Δ > 0. | VI: modelado temporal (Transformer con MIL / sin modelado temporal). VD: clasificación y localización. | Diferencia de F1-macro, de AUC por frame y de AP por frame. |
| PE4: ¿Qué componente de la arquitectura explica el desempeño: la representación de movimiento, el backbone de apariencia o el filtro de calidad de datos? | OE4: Hacer ablaciones que cambian un solo componente del modelo propuesto. | HE4: El flujo óptico de Farnebäck mejora el reconocimiento de agresiones frente a la diferencia de frames, y un preentrenamiento CLIP en apariencia mejora el F1-macro y el F1 de Aggression. H0: Δ ≤ 0. H1: Δ > 0 (variante − M0). | VI: configuración (V1 Farnebäck, V2 sin filtro de brillo, V3 ViT-B/16 CLIP, V4 ConvNeXt-B CLIP). VD: clasificación y localización. | Diferencia de F1-macro, F1 de Aggression y AUC por frame frente a M0; bootstrap pareado y McNemar. |

## 2. Protocolo de comparación

1. Mismos datos: videos de UCF-Crime a 320×240, 1 de cada 10 frames, filtro de brillo de 30, clases Normal, Aggression (Assault, Abuse, Fighting) y Theft (Robbery, Burglary, Stealing, Shoplifting), particiones v2 con huellas SHA-256. El test es el oficial de UCF-Crime.
2. Features: el modelo propuesto usa ConvNeXt-S sobre RGB y ConvNeXt-T sobre |Δframe|, congelados. Sultani, RTFM y UR-DMU usan I3D, como en sus artículos, extraído de los mismos videos (10 recortes, 16 frames por snippet). La regresión logística usa las mismas features del modelo propuesto.
3. Tres clases en los métodos de referencia: se les agrega una cabeza de tres clases sobre sus propios segmentos top-k, con pérdida L = L_original + 1,0 · CE. Sus hiperparámetros son los de sus repositorios oficiales.
4. Mismas métricas y el mismo código para todos: `compare.py` (F1, AUC y AP por video, bootstrap y McNemar), `frame_auc.py` y `vad localize` (AUC, AP y tIoU por frame).
5. Selección de modelo solo con validación (mejor F1-macro de validación); el test se usa una sola vez por modelo.
6. Estadística: cinco semillas por modelo; bootstrap estratificado y pareado por video (2000 remuestreos, IC 95 %) y McNemar exacto sobre el promedio de probabilidades de las semillas; α = 0,05.

## 3. Métodos de referencia y valores publicados

| Método | Publicación | Idea | Salida original | Publicado (protocolo oficial, 13 clases) | Hipótesis |
|---|---|---|---|---|---|
| Sultani et al. (MIL ranking) | CVPR 2018 | MLP de 3 capas; ranking MIL entre el segmento más anómalo de un video anómalo y el de uno normal | Binaria | AUC 75,41 (C3D) | HE1, HE2 |
| RTFM | ICCV 2021 | Magnitud de las features: los segmentos anómalos tienen mayor norma; top-k + red temporal multiescala | Binaria | AUC 84,30 (I3D) | HE1, HE2 |
| UR-DMU | AAAI 2023 | Memorias duales (normal y anómala) con regulación de incertidumbre + Transformer global-local | Binaria | AUC 86,97 (I3D) | HE1, HE2 |
| Regresión logística (B0) | Baseline clásico | Logística sobre el promedio temporal de las mismas features del modelo propuesto; por snippet para la localización | Tres clases | — | HE3 |
| Modelo propuesto (M0) | Esta tesis | Dos flujos (ConvNeXt-S + ConvNeXt-T sobre \|Δframe\|) → Transformer temporal + attention pooling + scorer local, pérdidas MIL | Tres clases + puntuación por snippet | — | Todas |

Los valores publicados son solo de contexto: usan 13 categorías y otra resolución. La comparación de la tesis es la reimplementación con nuestro protocolo.

## 4. Variables intervinientes y de control

| Variable | Tipo | Cómo se controla |
|---|---|---|
| Particiones train / val / test | Control | Particiones v2 por video, versionadas con hash; test oficial de 203 videos, sin fuga |
| Resolución y muestreo de frames | Control | 320×240, 1 de cada 10 frames |
| Clases | Control | Normal / Aggression / Theft; versión binaria = Normal frente al resto |
| Semilla aleatoria | Control | Cinco semillas por modelo; media ± desviación estándar |
| Selección de hiperparámetros | Control | Solo con validación; métodos de referencia con sus valores oficiales; búsqueda con Optuna sobre M0 |
| Hardware | Control | RTX 5070 Ti de 16 GB, Windows 11 |
| Features de entrada | Interviniente | M0 y B0 usan ConvNeXt; los métodos de referencia usan I3D, como en sus artículos. Se reporta como limitación |
| Desbalance de clases | Interviniente | Pesos de clase en la pérdida; métricas macro (test: 150 / 10 / 43) |
| Calidad de las anotaciones temporales | Interviniente | Anotaciones oficiales de UCF-Crime; se reporta como limitación |

## 5. Resultados frente a la matriz

Test, 203 videos. Media de cinco semillas, salvo B0, que es determinista (una corrida). El detalle está en `03_experimentos.ipynb` y en el documento del capítulo IV.

| Modelo | F1-macro | F1-anomalía | AUC por video | AP por video | AUC por frame | AP por frame | tIoU medio |
|---|---|---|---|---|---|---|---|
| Sultani et al. | 0,566 ± 0,036 | 0,623 | 0,855 | 0,710 | 0,858 | ver notebook 03 | — |
| RTFM | 0,520 ± 0,043 | 0,589 | 0,824 | 0,630 | 0,824 | ver notebook 03 | — |
| UR-DMU | 0,622 ± 0,027 | 0,666 | 0,878 | 0,720 | 0,883 (semilla 1) | ver notebook 03 | — |
| Regresión logística (B0) | 0,806 | 0,825 | 0,948 | 0,912 | 0,878 | 0,298 | — |
| Modelo propuesto (M0) | 0,689 ± 0,032 | 0,794 | 0,937 | 0,887 | 0,902 | 0,444 | 0,218 ± 0,013 |

| Hipótesis | Resultado | ¿Se acepta H1? |
|---|---|---|
| HE1: F1-macro de M0 mayor que cada referencia | Frente a Sultani (+0,123; p = 0,003) y RTFM (+0,169; p < 0,001), con McNemar p < 0,001. Frente a UR-DMU, +0,066 con un intervalo que roza el cero (p = 0,06), aunque M0 gana en F1-anomalía, AUC por video y McNemar (p = 0,001) | Sí con Sultani y RTFM; parcial con UR-DMU |
| HE2: AUC por frame de M0 igual o mayor | 0,902 frente a 0,858, 0,824 y 0,883 | Sí |
| HE3: el Transformer con MIL supera a la logística | En clasificación no: B0 saca 0,806 frente a 0,689 (bootstrap p < 0,001; McNemar 6 / 6, p = 1,00). En localización sí: AUC por frame 0,902 frente a 0,878 y AP 0,444 frente a 0,298 | No en clasificación; sí en localización |
| HE4: Farnebäck mejora agresiones; CLIP mejora F1-macro y Aggression | Farnebäck: Aggression 0,456 frente a 0,421 (p = 0,53) y peor localización. ViT-B/16 CLIP: F1-macro 0,764 (p = 0,016) y Aggression 0,602 (p = 0,014). ConvNeXt-B CLIP: 0,728 (p = 0,17) | No para Farnebäck; sí para ViT-CLIP; no para ConvNeXt-B CLIP |

Que HE3 no se cumpla en clasificación es un resultado, no un error: indica que el poder para reconocer el tipo de delito ya está en las features de los dos flujos y que la cabeza temporal aporta sobre todo la ubicación del evento.